# VAYU v2 — Western Ghats Climate Digital Twin
## ISRO BAH 2026 | GATv2 + Weighted CRPS + 45-day window

**Required Kaggle input datasets (Add → Dataset):**
1. `shyam31415/vayu-western-ghats-processed-v1` ← IMD data
2. `shyam31415/vayu-ancillary-wg-v1` ← NCEP wind + CHIRPS + GEBCO

**Optional (for warm-start):**
3. `shyam31415/vayu-v1-best` ← upload `vayu_best (3).pt` here to enable `WARM_V1` mode

---
### Training modes
| Mode | Description | When to use |
|------|-------------|-------------|
| `FRESH_V2` | Fresh v2 model (9.5M params, hidden=192, d=384, 6L-Transformer) | **Stage 1** — default |
| `WARM_V1` | Continue from `vayu_best (3).pt` (2.3M, R²=0.200) with v1 arch + v2 loss | If you want faster improvement past 0.200 |
| `WARM_V2` | Continue from a saved v2 checkpoint | **Stages 2–4** |

> **Why `WARM_V1` uses v1 architecture dimensions (hidden=128):**
> The v1 checkpoint has hidden=128 while v2 default is hidden=192. Shapes are incompatible.
> The loader auto-sets the model to match the checkpoint's dims, then remaps encoder keys
> (`encoder.input_proj.X` → `encoder.X`). GAT + Transformer weights also transfer cleanly.
> Heads (rainfall two-stage, tmax, tmin) re-initialise from scratch with the better VayuV2Loss.

---
### Bugs fixed vs Session 1
- ❌ Phase curriculum → loss-scale mismatch → no Phase 2 checkpoints saved → **REMOVED**
- ❌ Tweedie on z-scores (requires y≥0, breaks on negative normalised values) → **REPLACED with CRPS+BCE**
- ✅ Dual checkpoint: saves on `val_loss` improvement **OR** `R²_rain` improvement
- ✅ Model architecture auto-adapts to checkpoint dimensions in warm-start mode

---
### Aurora baseline (optional)
Cell 12 runs `microsoft-aurora` inference as a comparison baseline.
Disabled by default (`RUN_AURORA = False`) — enable only **after** training completes
to avoid eating the 12 hr GPU budget.


In [ ]:

# ── 1. GPU check + install deps ───────────────────────────────────────────────
import subprocess, sys, os, shutil, torch

# nvidia-smi only exists when a GPU accelerator is attached
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
else:
    print('⚠ nvidia-smi not found.')
    print('  → Go to  Session options → Accelerator → GPU T4 x2  then Save & Run All.')
    print('  Training on CPU would take ~50× longer and will likely time-out.')

print('Python:', sys.version)
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    print('\n‼ NO GPU DETECTED — stopping now to avoid wasting session time.')
    print('  Enable a GPU accelerator in Session options and restart the session.')
    raise SystemExit('Add a GPU accelerator (T4×2 or P100) before running.')

!pip install -q torch-geometric==2.5.3 xarray netcdf4 typer scipy
print('✓ Dependencies installed')

# ══════════════════════════════════════════════════════════════════════════════
# TRAINING MODE  ← change here before each stage
# ══════════════════════════════════════════════════════════════════════════════
TRAINING_MODE = 'WARM_V1'   # FRESH_V2 | WARM_V1 | WARM_V2

# How many NEW epochs to add on top of the warm-start checkpoint
# WARM_V1:  vayu_best (3).pt is ep27 → runs ep 28–52 (25 new epochs ≈ 7 hrs on T4)
# WARM_V2:  set to 25 for Stage 3, 50 for Stage 4, etc.
WARM_EXTRA_EPOCHS = 25

# Used only for FRESH_V2 (fresh from scratch)
FRESH_TOTAL_EPOCHS = 25

assert TRAINING_MODE in ('WARM_V1', 'WARM_V2', 'FRESH_V2', 'FRESH_V1')
print(f'\nTraining mode : {TRAINING_MODE}  |  Extra epochs: {WARM_EXTRA_EPOCHS}')
print(f'Target: R²_rain > 0.40  |  R²_tmax > 0.90')
print(f'Checkpoint → /kaggle/working/vayu_best.pt on every improvement')


In [ ]:

# ── 2. Mount repo + locate Kaggle datasets ────────────────────────────────────
import sys, os, shutil
from pathlib import Path

REPO_DIR = '/kaggle/working/isro'
os.makedirs(f'{REPO_DIR}/checkpoints/wg_v2', exist_ok=True)

if os.path.exists(f'{REPO_DIR}/.git'):
    os.system(f'git -C {REPO_DIR} pull --quiet')
else:
    os.system(f'rm -rf {REPO_DIR}')
    os.system(f'git clone --quiet https://github.com/Shyamistic/vayu.git {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo:', os.getcwd())

root = Path('/kaggle/input')

# IMD processed dataset
DATASET_DIR = None
for nc in root.rglob('normalized_2010-2025.nc'):
    DATASET_DIR = str(nc.parent); break
if not DATASET_DIR:
    raise RuntimeError("vayu-western-ghats-processed-v1 not found — add via Add Input")
print('IMD dataset:', DATASET_DIR)

# Ancillary (NCEP wind, CHIRPS, GEBCO)
ANCD_DIR = None
for nc in root.rglob('gebco_*.nc'):
    ANCD_DIR = str(nc.parent); break
print('Ancillary   :', ANCD_DIR or '⚠ NOT FOUND — add vayu-ancillary-wg-v1')

# Warm-start checkpoint (only needed for WARM_V2/WARM_V1)
V2_CKPT_PATH = None
if TRAINING_MODE in ('WARM_V1','WARM_V2'):
    for pt in sorted(root.rglob('*.pt')):
        if pt.stat().st_size > 1e6:
            V2_CKPT_PATH = str(pt); break
    print('Checkpoint  :', V2_CKPT_PATH or '⚠ NOT FOUND — will fall back to FRESH')
    if not V2_CKPT_PATH:
        TRAINING_MODE = 'FRESH_V2' if 'V2' in TRAINING_MODE else 'FRESH_V1'
        print(f'→ Falling back to {TRAINING_MODE}')


In [ ]:

# ── 3. Copy files + NCEP enrichment + build sequences ────────────────────────
import subprocess, sys, shutil, os
from pathlib import Path
import xarray as xr
import numpy as np

PY = sys.executable
PROCESSED_DIR = f'{REPO_DIR}/data/processed_western_ghats'
NCEP_DST      = f'{REPO_DIR}/data/ncep_wind_subset'
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(NCEP_DST, exist_ok=True)

# Copy IMD processed files
for f in Path(DATASET_DIR).glob('*'):
    dst = Path(PROCESSED_DIR) / f.name
    if not dst.exists(): shutil.copy2(f, dst)
print('✓ IMD processed files copied')

# Copy ancillary (NCEP wind, CHIRPS, GEBCO)
if ANCD_DIR:
    n_ncep = 0
    for pat in ['uwnd_*.nc','vwnd_*.nc','shum_*.nc','pr_wtr_*.nc']:
        for f in Path(ANCD_DIR).glob(pat):
            shutil.copy2(f, NCEP_DST); n_ncep += 1
    for f in Path(ANCD_DIR).glob('chirps_*.nc'):
        dst_d = Path(REPO_DIR)/'chirps'; dst_d.mkdir(exist_ok=True)
        shutil.copy2(f, dst_d)
    for f in Path(ANCD_DIR).glob('gebco_*.nc'):
        dst_d = Path(REPO_DIR)/'gebco'; dst_d.mkdir(exist_ok=True)
        shutil.copy2(f, dst_d)
    print(f'✓ Ancillary copied: {n_ncep} NCEP files | CHIRPS | GEBCO')

# NCEP enrichment — add uwnd_850, vwnd_850, shum_850 to normalized file
NORM_FILE = f'{PROCESSED_DIR}/normalized_2010-2025.nc'
ds_check  = xr.open_dataset(NORM_FILE)
existing  = list(ds_check.data_vars); ds_check.close()
needs_ncep = not any('uwnd' in v or 'vwnd' in v for v in existing)
print(f'\nNormalized vars: {existing}')
print(f'NCEP enrichment needed: {needs_ncep}')

if needs_ncep and any(Path(NCEP_DST).glob('uwnd_*.nc')):
    print('Enriching with NCEP 850 hPa wind + humidity...')
    ds = xr.open_dataset(NORM_FILE)
    lats, lons, times = ds.lat.values, ds.lon.values, ds.time.values
    added = []
    for varname, pattern in [('uwnd_850','uwnd_*.nc'),('vwnd_850','vwnd_*.nc'),('shum_850','shum_*.nc')]:
        files = sorted(Path(NCEP_DST).glob(pattern))
        if not files or varname in ds.data_vars: continue
        try:
            # ── Interpolate each year to the TARGET grid BEFORE concat ──────
            # Concat with join='override' fails when lat-sizes differ across years.
            rg_parts = []
            for f in files:
                d   = xr.open_dataset(f)
                rv  = list(d.data_vars)[0]
                arr = d[rv]
                # Normalise dim names
                rmap = {dim:'lat' for dim in arr.dims if dim.lower() in ('latitude','nav_lat')}
                rmap.update({dim:'lon' for dim in arr.dims if dim.lower() in ('longitude','nav_lon')})
                if rmap: arr = arr.rename(rmap)
                # Interpolate this year to target grid (handles lat-size mismatches)
                try:
                    rg_yr = arr.interp({'lat':lats,'lon':lons}, method='linear').fillna(0.0)
                except Exception:
                    rg_yr = arr.interp({'lat':lats,'lon':lons}, method='nearest').fillna(0.0)
                rg_parts.append(rg_yr); d.close()
            # All parts now share the same lat/lon → concat is safe
            comb = xr.concat(rg_parts, dim='time').sortby('time')
            rg   = comb.interp(time=times, method='nearest').fillna(0.0)
            mean_v = float(rg.values.mean())
            std_v  = max(float(rg.values.std()), 1e-6)
            ds[varname] = (rg - mean_v) / std_v
            added.append(varname)
            print(f'  ✓ {varname}: {len(files)} years | mean={mean_v:.3f} std={std_v:.3f}')
        except Exception as e:
            print(f'  ⚠ {varname}: {e}')
    if added:
        tmp = NORM_FILE + '.tmp'; ds.to_netcdf(tmp); ds.close(); shutil.move(tmp, NORM_FILE)
        ds2 = xr.open_dataset(NORM_FILE)
        print(f'✓ Enriched vars: {list(ds2.data_vars)}'); ds2.close()
    else:
        ds.close()

# Build sequences: 1200 train / 200 val
print('\n=== Building sequences (45d window, 1200 train / 200 val) ===')
r = subprocess.run(
    [PY, '-m', 'data_ingestion.cli', 'build-sequences',
     '--normalized-file', NORM_FILE, '--input-window', '45',
     '--target-window', '7', '--max-train', '1200',
     '--max-val', '200', '--stride', '3', '--output-dir', PROCESSED_DIR],
    capture_output=True, text=True, cwd=REPO_DIR
)
if r.returncode == 0:
    print(r.stdout[-600:]); print('✓ Sequences built')
else:
    print('⚠ build-sequences failed'); print(r.stderr[-500:])

for f in sorted(Path(PROCESSED_DIR).glob('*.pt')):
    print(f'  {f.name}: {f.stat().st_size/1e6:.0f} MB')


In [ ]:

# ── 4. Model architecture: VayuClimateModelV2 ─────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

class GATv2Block(nn.Module):
    def __init__(self, hidden: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert hidden % heads == 0
        self.gat  = GATv2Conv(hidden, hidden // heads, heads=heads, dropout=dropout, add_self_loops=True)
        self.norm  = nn.LayerNorm(hidden)
        self.ff    = nn.Sequential(nn.Linear(hidden, hidden*2), nn.GELU(), nn.Linear(hidden*2, hidden))
        self.norm2 = nn.LayerNorm(hidden)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x, edge_index):
        h = self.norm(x + self.drop(self.gat(x, edge_index)))
        return self.norm2(h + self.drop(self.ff(h)))

class TwoStageRainfallHead(nn.Module):
    def __init__(self, d_model: int, target_steps: int):
        super().__init__()
        self.occurrence = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps))
        self.amount     = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps))
        self.sigma      = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, target_steps), nn.Softplus())
    def forward(self, z):
        return self.occurrence(z), self.amount(z), self.sigma(z).clamp(0.01, 5.0)

class VayuClimateModelV2(nn.Module):
    def __init__(self, in_channels: int = 6, gnn_hidden: int = 192, d_model: int = 384,
                 n_heads: int = 4, n_gat_layers: int = 4, n_transformer_layers: int = 6,
                 target_steps: int = 7, seq_len: int = 45, n_nodes: int = 0,
                 dropout: float = 0.1):
        super().__init__()
        self.gnn_hidden   = gnn_hidden
        self.d_model      = d_model
        self.target_steps = target_steps

        self.encoder   = nn.Sequential(nn.Linear(in_channels, gnn_hidden), nn.LayerNorm(gnn_hidden), nn.SiLU())
        self.gat_layers = nn.ModuleList([GATv2Block(gnn_hidden, n_heads, dropout) for _ in range(n_gat_layers)])
        self.temporal_proj = nn.Linear(gnn_hidden * seq_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
                                               dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_transformer_layers)
        self.rain_head = TwoStageRainfallHead(d_model, target_steps)
        self.tmax_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, target_steps))
        self.tmin_head = nn.Sequential(nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, target_steps))

    def forward(self, x, edge_index, batch_map=None):
        B, T, N, F = x.shape

        # ── Properly tile edge_index across batch so every sample gets graph attention
        # (without this, only sample 0 gets edges; samples 1..B-1 only see self-loops)
        if edge_index.numel() > 0:
            offsets = torch.arange(B, device=x.device) * N
            ei_b = (edge_index.unsqueeze(0) + offsets.view(B, 1, 1)).reshape(2, -1)
        else:
            ei_b = edge_index

        # ── Encode + GATv2 per timestep ─────────────────────────────────────
        h = self.encoder(x.reshape(B*T*N, F)).reshape(B, T, N, self.gnn_hidden)
        gat_out = []
        for t in range(T):
            xt = h[:, t].reshape(B*N, self.gnn_hidden)         # contiguous slice
            for gat in self.gat_layers:
                xt = gat(xt, ei_b)
            gat_out.append(xt.reshape(B, N, self.gnn_hidden))

        # ── Temporal projection → Transformer ──────────────────────────────
        gat_seq  = torch.stack(gat_out, dim=1)                  # (B, T, N, H)
        node_seq = gat_seq.permute(0, 2, 1, 3).reshape(B*N, T*self.gnn_hidden)
        node_emb = F.relu(self.temporal_proj(node_seq)).reshape(B, N, self.d_model)
        ctx = self.transformer(node_emb)                         # (B, N, d_model)

        occ, mu, sigma = self.rain_head(ctx)
        tmax = self.tmax_head(ctx)
        tmin = self.tmin_head(ctx)
        return occ, mu, sigma, tmax, tmin

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Sanity forward pass
_m  = VayuClimateModelV2(in_channels=6, gnn_hidden=192, d_model=384).to(DEVICE)
_x  = torch.randn(2, 45, 10, 6, device=DEVICE)
_ei = torch.zeros(2, 0, dtype=torch.long, device=DEVICE)
with torch.no_grad():
    o, mu, s, tmax, tmin = _m(_x, _ei)
print(f'Forward pass OK: occ={tuple(o.shape)} mu={tuple(mu.shape)} tmax={tuple(tmax.shape)}')
print(f'Parameters: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M')
del _m, _x, _ei


In [ ]:

# ── 5. VayuV2Loss — weighted CRPS + BCE (NO Tweedie) ─────────────────────────
# Tweedie requires y≥0 but z-score normalized rainfall goes negative on dry days
# → val_loss spikes to 202, R²_rain = -0.35.  Fixed: CRPS + BCE only.

import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedCRPSLoss(nn.Module):
    """CRPS for probabilistic amount prediction (log-normal parametrization)."""
    def __init__(self, alpha: float = 3.0, heavy_threshold: float = 20.0):
        super().__init__()
        self.alpha = alpha
        self.heavy_threshold = heavy_threshold  # mm/day (in normalized space ~1.5σ)
    def forward(self, mu, sigma, target):
        """mu, sigma: (B,N,T)  target: (B,N,T) z-scored rainfall"""
        sigma = sigma.clamp(min=1e-6)
        z = (target - mu) / sigma
        normal = torch.distributions.Normal(0, 1)
        phi_z  = normal.log_prob(z).exp()
        Phi_z  = normal.cdf(z)
        crps   = sigma * (z * (2*Phi_z - 1) + 2*phi_z - 1/torch.pi**0.5)
        heavy_mask = (target > self.heavy_threshold / 30.0).float()   # rough z threshold
        weights = 1.0 + (self.alpha - 1.0) * heavy_mask
        return (crps * weights).mean()

class VayuV2Loss(nn.Module):
    """
    VAYU v2 loss — scientifically correct for zero-inflated precipitation.
    = 70% weighted CRPS (amount, log-normal)
    + 30% BCE       (occurrence, binary rain/no-rain)
    """
    def __init__(self):
        super().__init__()
        self.crps  = WeightedCRPSLoss(alpha=3.0)
        self.rain_w = 1.8  # down-weight balanced with tmax/tmin

    def forward(self, occ, mu, sigma, tmax_pred, tmin_pred,
                rain_true, tmax_true, tmin_true):
        # Occurrence: binary BCE — 0 = dry (< 1mm raw)
        DRY_THRESH = -0.3       # approximate z-score for 1mm/day
        rain_bin   = (rain_true > DRY_THRESH).float()
        bce        = F.binary_cross_entropy_with_logits(occ, rain_bin)
        # Amount: CRPS on log-normal distribution
        crps_loss  = self.crps(mu, sigma, rain_true)
        # Deterministic heads: MSE
        tmax_loss  = F.mse_loss(tmax_pred, tmax_true)
        tmin_loss  = F.mse_loss(tmin_pred, tmin_true)
        # Combined
        rain_total = 0.7 * crps_loss + 0.3 * bce
        total = (self.rain_w * rain_total + tmax_loss + tmin_loss) / (self.rain_w + 2.0)
        return total, {'crps': crps_loss.item(), 'bce': bce.item(),
                       'tmax': tmax_loss.item(), 'tmin': tmin_loss.item()}

# Quick test
criterion = VayuV2Loss()
_occ  = torch.randn(2, 10, 7)
_mu   = torch.randn(2, 10, 7)
_sig  = torch.ones(2, 10, 7)
_rain = torch.randn(2, 10, 7)
_loss, _comps = criterion(_occ, _mu, _sig, _mu, _mu, _rain, _rain, _rain)
print(f'VayuV2Loss sanity: {_loss.item():.4f}  comps={_comps}')
assert _loss.item() < 100, 'Loss too large — check CRPS/BCE'
print('✓ Loss functions OK')


In [ ]:

# ── 6. Checkpoint loader + architecture configuration ─────────────────────────
# WARM_V1  → vayu_best (3).pt  (CLI v1 format: top-level epoch/val_loss keys)
# WARM_V2  → saved v2 notebook checkpoint  (metadata dict inside payload)
# FRESH_V2 → no checkpoint; fresh v2 model with hidden=192, d=384, 6L
# ─────────────────────────────────────────────────────────────────────────────

def _load_raw_sd(ckpt_path):
    """Load checkpoint; normalise the two possible save formats:
       - v1 CLI:      {'model_state_dict':…, 'config':…, 'epoch':N, 'val_loss':X}
       - v2 notebook: {'model_state_dict':…, 'optimizer_state_dict':…, 'metadata':{…}}
    """
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if not (isinstance(raw, dict) and 'model_state_dict' in raw):
        # bare state dict (unlikely but safe fallback)
        return raw, {}

    sd = raw['model_state_dict']

    # Try v2 notebook format first
    meta = raw.get('metadata', {})
    if meta:
        return sd, meta

    # v1 CLI format: epoch / val_loss sit at the top level
    meta = {}
    for k in ('epoch', 'val_loss', 'r2_rain', 'r2_tmax', 'best_r2_rain', 'best_val_loss'):
        if k in raw:
            meta[k] = raw[k]

    # Also try to extract hidden_dim from the saved ModelConfig if present
    cfg = raw.get('config')
    if cfg is not None:
        try:
            meta.setdefault('model_hidden', getattr(cfg, 'gnn_hidden_dim', None))
            meta.setdefault('model_dmodel', getattr(cfg, 'd_model', None))
        except Exception:
            pass

    return sd, meta

def _is_v1_checkpoint(sd):
    """v1 uses  encoder.input_proj.X  keys;  v2 uses  encoder.X"""
    return 'encoder.input_proj.0.weight' in sd

def _remap_v1_to_v2(sd):
    """Rename encoder.input_proj.X → encoder.X so v2 nn.Sequential picks them up."""
    out = {}
    for k, v in sd.items():
        if k.startswith('encoder.input_proj.'):
            new_k = 'encoder.' + k[len('encoder.input_proj.'):]
            out[new_k] = v
        else:
            out[k] = v   # GAT / Transformer / head keys are structurally identical
    return out

def _detect_dims(sd):
    """Read hidden_dim and d_model straight from weight shapes."""
    enc = sd.get('encoder.0.weight') or sd.get('encoder.input_proj.0.weight')
    hidden  = int(enc.shape[0]) if enc is not None else 192
    tp      = sd.get('temporal_proj.weight')
    d_model = int(tp.shape[0]) if tp is not None else 384
    n_layers = sum(1 for k in sd
                   if k.startswith('transformer.layers.')
                   and k.endswith('.self_attn.in_proj_weight'))
    n_layers = n_layers if n_layers > 0 else 6
    return hidden, d_model, n_layers

# ── Defaults (used if FRESH) ───────────────────────────────────────────────────
MODEL_HIDDEN = 192
MODEL_DMODEL = 384
MODEL_NL     = 6
WARM_SD      = None
START_EPOCH  = 0
TOTAL_EPOCHS = FRESH_TOTAL_EPOCHS

# ── Warm-start handling ────────────────────────────────────────────────────────
if TRAINING_MODE in ('WARM_V1', 'WARM_V2') and V2_CKPT_PATH:
    sd, meta = _load_raw_sd(V2_CKPT_PATH)
    is_v1    = _is_v1_checkpoint(sd)
    print(f'Checkpoint : {V2_CKPT_PATH}')
    print(f'Format     : {"v1 CLI (remapping encoder keys)" if is_v1 else "v2 notebook"}')
    print(f'Epoch      : {meta.get("epoch","?")}')
    print(f'val_loss   : {meta.get("val_loss","?")}')
    print(f'R²_rain    : {meta.get("r2_rain","N/A")}   R²_tmax: {meta.get("r2_tmax","N/A")}')

    if is_v1:
        sd = _remap_v1_to_v2(sd)

    MODEL_HIDDEN, MODEL_DMODEL, MODEL_NL = _detect_dims(sd)
    WARM_SD      = sd
    START_EPOCH  = int(meta.get('epoch', 0))
    TOTAL_EPOCHS = START_EPOCH + WARM_EXTRA_EPOCHS

    print(f'\nAuto-detected  hidden={MODEL_HIDDEN}  d_model={MODEL_DMODEL}  n_layers={MODEL_NL}')
    print(f'→ Warm-start ep {START_EPOCH} → {TOTAL_EPOCHS}  ({WARM_EXTRA_EPOCHS} new epochs)')

    if is_v1:
        n_enc   = sum(1 for k in WARM_SD if k.startswith('encoder.'))
        n_gat   = sum(1 for k in WARM_SD if 'gat_layers' in k)
        n_trans = sum(1 for k in WARM_SD if 'transformer' in k)
        print(f'Transferable: encoder×{n_enc}  GAT×{n_gat}  Transformer×{n_trans}')
        print(f'Re-init from scratch: rain_head (TwoStage), tmax_head, tmin_head')

elif TRAINING_MODE in ('FRESH_V2', 'FRESH_V1'):
    print(f'→ FRESH start  hidden={MODEL_HIDDEN}  d_model={MODEL_DMODEL}  n_layers={MODEL_NL}')
    print(f'  Epochs 1–{TOTAL_EPOCHS}')

else:
    print(f'⚠ WARM mode requested but no checkpoint found → FRESH_V2 fallback')

print(f'\nNew epochs to train: {TOTAL_EPOCHS - START_EPOCH}')


In [ ]:

# ── 7. Data loading ───────────────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

PROCESSED_DIR = f'{REPO_DIR}/data/processed_western_ghats'

def _convert_old_seqs(seqs):
    """Convert CLI old-format list[(GraphData, target)] → flat tensors.

    CLI build-sequences saves (GraphData, tensor) tuples where:
      graph.x  : (N_nodes, seq_len, features)   → want X: (B, seq_len, N_nodes, F)
      target   : (horizon, N_nodes, 3)            → want y: (B, N_nodes, 3*horizon)
                  last dim = [rain, tmax, tmin]            [rain*H, tmax*H, tmin*H]
    """
    graphs, targets = zip(*seqs)
    # (B, N, T, F) → (B, T, N, F)
    X = torch.stack([g.x for g in graphs], dim=0).permute(0, 2, 1, 3).contiguous()
    t = torch.stack(targets, dim=0)           # (B, H, N, 3)
    rain = t[..., 0].permute(0, 2, 1)         # (B, N, H)
    tmax = t[..., 1].permute(0, 2, 1)
    tmin = t[..., 2].permute(0, 2, 1)
    y  = torch.cat([rain, tmax, tmin], dim=-1).contiguous()  # (B, N, 3*H)
    ei = graphs[0].edge_index
    return X, y, ei

def load_sequences(processed_dir: str):
    """Load sequences — handles both flat-tensor and old GraphData-list formats."""
    d = Path(processed_dir)

    if (d / 'X_train.pt').exists():
        # ── New flat-tensor format ─────────────────────────────────────────
        X_tr  = torch.load(d / 'X_train.pt', weights_only=False)
        y_tr  = torch.load(d / 'y_train.pt', weights_only=False)
        X_val = torch.load(d / 'X_val.pt',   weights_only=False)
        y_val = torch.load(d / 'y_val.pt',   weights_only=False)
        graph = torch.load(d / 'graph.pt',   weights_only=False)
        ei    = graph.edge_index if hasattr(graph, 'edge_index') \
                else graph.get('edge_index', torch.zeros(2, 0, dtype=torch.long))

    elif (d / 'train_sequences.pt').exists():
        # ── Old CLI format (GraphData list) — convert on the fly ──────────
        print('Converting CLI sequence format → flat tensors (may take ~30s)...')
        tr         = torch.load(d / 'train_sequences.pt', weights_only=False)
        X_tr, y_tr, ei = _convert_old_seqs(tr); del tr
        val        = torch.load(d / 'val_sequences.pt',   weights_only=False)
        X_val, y_val, _ = _convert_old_seqs(val); del val
        print(f'  X_tr={tuple(X_tr.shape)}  y_tr={tuple(y_tr.shape)}')

    else:
        raise FileNotFoundError(
            f'No sequence files found in {processed_dir}. '
            'Expected X_train.pt or train_sequences.pt')

    print(f'Train: {X_tr.shape}  Val: {X_val.shape}')
    print(f'Nodes: {X_tr.shape[2]}  Features: {X_tr.shape[3]}  '
          f'Edge pairs: {ei.shape[1] if ei.numel() else 0}')
    return X_tr, y_tr, X_val, y_val, ei

def unpack_seq(y_batch):
    """y: (B, N, 3*T) → rain (B,N,T), tmax (B,N,T), tmin (B,N,T)"""
    T = y_batch.shape[-1] // 3
    return y_batch[:,:,:T], y_batch[:,:,T:2*T], y_batch[:,:,2*T:]

X_tr, y_tr, X_val, y_val, EDGE_INDEX = load_sequences(PROCESSED_DIR)
EDGE_INDEX = EDGE_INDEX.to(DEVICE)

N_NODES   = X_tr.shape[2]
IN_FEAT   = X_tr.shape[3]
SEQ_LEN   = X_tr.shape[1]
TARGET_T  = y_tr.shape[-1] // 3
BATCH_SIZE = 4

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE,
                          shuffle=True,  pin_memory=True, drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE,
                          shuffle=False, pin_memory=True)
print(f'Batches — train: {len(train_loader)}  val: {len(val_loader)}')
print(f'Config  — N_NODES:{N_NODES} IN_FEAT:{IN_FEAT} SEQ_LEN:{SEQ_LEN} TARGET_T:{TARGET_T}')


In [ ]:

# ── 8. Training loop ──────────────────────────────────────────────────────────
# Persistence guarantees (survive mid-session cancellation):
#   vayu_best.pt  – best model so far (val_loss OR R²_rain improved)
#   vayu_last.pt  – most recent completed epoch (always overwritten)
#   training_log.csv – one row per epoch, flushed immediately after each epoch
# All writes are atomic: save to .tmp then os.replace → no partial files.

import math, time, csv, os

def _atomic_save(payload, path):
    """Write to .tmp then atomic rename — safe against mid-write cancellation."""
    tmp = path + '.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, path)            # atomic on Linux (Kaggle)

LOG_FILE  = '/kaggle/working/training_log.csv'
CKPT_ROOT = '/kaggle/working/vayu_best.pt'
CKPT_LAST = '/kaggle/working/vayu_last.pt'
CKPT_DIR  = f'{REPO_DIR}/checkpoints/wg_v2'

# ── Open log CSV (append so it survives re-runs) ──────────────────────────────
log_is_new = not os.path.exists(LOG_FILE)
log_fh = open(LOG_FILE, 'a', newline='', buffering=1)   # line-buffered
log_writer = csv.writer(log_fh)
if log_is_new:
    log_writer.writerow(['epoch','train_loss','val_loss','r2_rain','r2_tmax','lr','elapsed_s','saved'])

# ── Build model ────────────────────────────────────────────────────────────────
print(f'Building VayuClimateModelV2  hidden={MODEL_HIDDEN}  d_model={MODEL_DMODEL}  n_layers={MODEL_NL}')
model = VayuClimateModelV2(
    in_channels         = IN_FEAT,
    gnn_hidden          = MODEL_HIDDEN,
    d_model             = MODEL_DMODEL,
    n_heads             = 4,
    n_gat_layers        = 4,
    n_transformer_layers= MODEL_NL,
    target_steps        = TARGET_T,
    seq_len             = SEQ_LEN,
    n_nodes             = N_NODES,
).to(DEVICE)

if WARM_SD is not None:
    missing, unexpected = model.load_state_dict(WARM_SD, strict=False)
    loaded = len(WARM_SD) - len(missing)
    print(f'✓ Warm-start: {loaded}/{len(WARM_SD)} tensors loaded  |  re-init: {len(missing)}')
else:
    print('✓ Fresh random initialisation')

criterion = VayuV2Loss()
NEW_EPOCHS = TOTAL_EPOCHS - START_EPOCH
INIT_LR    = 1e-4 if WARM_SD is not None else 3e-4
optimizer  = torch.optim.AdamW(model.parameters(), lr=INIT_LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NEW_EPOCHS, eta_min=1e-6)

best_val_loss = float('inf')
best_r2_rain  = -float('inf')

params = sum(p.numel() for p in model.parameters())
print(f'\n{"─"*70}')
print(f'VayuClimateModelV2  |  {params/1e6:.2f}M params')
print(f'Epochs {START_EPOCH+1}–{TOTAL_EPOCHS}  ({NEW_EPOCHS} new)  |  '
      f'Batch {BATCH_SIZE}  |  LR {INIT_LR:.0e}')
print(f'Mode: {TRAINING_MODE}  |  Loss: VayuV2Loss (CRPS 70% + BCE 30%)')
print(f'Outputs: {CKPT_ROOT}  (best)  |  {CKPT_LAST}  (last epoch)')
print(f'         {LOG_FILE}  (per-epoch log, always flushed)')
print(f'{"─"*70}')

def r2_score(pred, true):
    ss_res = ((pred - true)**2).sum()
    ss_tot = ((true - true.mean())**2).sum()
    return float(1 - ss_res / (ss_tot + 1e-8))

scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda')

try:
    for epoch in range(START_EPOCH + 1, TOTAL_EPOCHS + 1):
        t0 = time.time()

        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        tr_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            rain_t, tmax_t, tmin_t = unpack_seq(yb)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
                loss, comps = criterion(occ, mu, sigma, tmax_p, tmin_p, rain_t, tmax_t, tmin_t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            tr_loss += loss.item()
        tr_loss /= len(train_loader)

        # ── Validation ─────────────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        rain_preds, rain_true_all = [], []
        tmax_preds, tmax_true_all = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                rain_t, tmax_t, tmin_t = unpack_seq(yb)
                with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                    occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
                    loss, _ = criterion(occ, mu, sigma, tmax_p, tmin_p, rain_t, tmax_t, tmin_t)
                val_loss += loss.item()
                rain_preds.append(mu.cpu()); rain_true_all.append(rain_t.cpu())
                tmax_preds.append(tmax_p.cpu()); tmax_true_all.append(tmax_t.cpu())
        val_loss /= len(val_loader)

        r2_rain = r2_score(torch.cat(rain_preds), torch.cat(rain_true_all))
        r2_tmax = r2_score(torch.cat(tmax_preds), torch.cat(tmax_true_all))
        scheduler.step()
        elapsed = time.time() - t0
        lr_now  = scheduler.get_last_lr()[0]

        print(f'Ep {epoch:3d}/{TOTAL_EPOCHS} | train={tr_loss:.4f} val={val_loss:.4f} '
              f'| R²_rain={r2_rain:.3f} R²_tmax={r2_tmax:.3f} | {elapsed:.0f}s | lr={lr_now:.1e}',
              flush=True)

        # ── Build checkpoint payload ───────────────────────────────────────────
        meta = {'epoch': epoch, 'val_loss': val_loss, 'r2_rain': r2_rain, 'r2_tmax': r2_tmax,
                'training_mode': TRAINING_MODE,
                'model_hidden': MODEL_HIDDEN, 'model_dmodel': MODEL_DMODEL, 'model_nl': MODEL_NL,
                'best_val_loss': best_val_loss, 'best_r2_rain': best_r2_rain}
        payload = {'model_state_dict': model.state_dict(),
                   'optimizer_state_dict': optimizer.state_dict(), 'metadata': meta}

        # ── Always save last epoch (atomic) ───────────────────────────────────
        _atomic_save(payload, CKPT_LAST)

        # ── Save best checkpoint (dual metric, atomic) ────────────────────────
        save_reason = []
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss; save_reason.append(f'val_loss↓{val_loss:.4f}')
        if r2_rain > best_r2_rain + 0.005:
            best_r2_rain = r2_rain; save_reason.append(f'R²_rain↑{r2_rain:.3f}')

        if save_reason:
            _atomic_save(payload, CKPT_ROOT)
            _atomic_save(payload, f'{CKPT_DIR}/vayu_v2_ep{epoch:03d}.pt')
            print(f'    ✓ Best checkpoint updated ({" | ".join(save_reason)})', flush=True)

        # ── Append to log CSV (flushed every row) ─────────────────────────────
        log_writer.writerow([epoch, f'{tr_loss:.5f}', f'{val_loss:.5f}',
                             f'{r2_rain:.4f}', f'{r2_tmax:.4f}',
                             f'{lr_now:.2e}', f'{elapsed:.1f}',
                             '✓' if save_reason else ''])

finally:
    log_fh.close()
    print(f'\n✓ Done  |  best_val_loss={best_val_loss:.4f}  best_R²_rain={best_r2_rain:.3f}')
    print(f'Logs saved to: {LOG_FILE}')
    print(f'Best model  : {CKPT_ROOT}')
    print(f'Last epoch  : {CKPT_LAST}')


In [ ]:

# ── 9. Evaluation + visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

print('=== Final Evaluation ===')
model.eval()
all_rain_p, all_rain_t = [], []
all_tmax_p, all_tmax_t = [], []
all_tmin_p, all_tmin_t = [], []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        rain_t, tmax_t, tmin_t = unpack_seq(yb)
        occ, mu, sigma, tmax_p, tmin_p = model(xb, EDGE_INDEX)
        all_rain_p.append(mu.cpu()); all_rain_t.append(rain_t.cpu())
        all_tmax_p.append(tmax_p.cpu()); all_tmax_t.append(tmax_t.cpu())
        all_tmin_p.append(tmin_p.cpu()); all_tmin_t.append(tmin_t.cpu())

rp = torch.cat(all_rain_p).flatten().numpy()
rt = torch.cat(all_rain_t).flatten().numpy()
tp = torch.cat(all_tmax_p).flatten().numpy()
tt = torch.cat(all_tmax_t).flatten().numpy()

def r2(p, t):
    return 1 - ((p-t)**2).sum() / (((t-t.mean())**2).sum() + 1e-8)

r2_rain = r2(rp, rt)
r2_tmax = r2(tp, tt)
print(f'R²_rain : {r2_rain:.4f}  (v1 baseline 0.200  →  v2 target 0.300)')
print(f'R²_tmax : {r2_tmax:.4f}  (v1 baseline 0.817)')

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].scatter(rt[:2000], rp[:2000], alpha=0.3, s=2, color='steelblue')
axs[0].set_xlabel('True rainfall (z-score)'); axs[0].set_ylabel('Predicted'); axs[0].set_title(f'Rainfall R²={r2_rain:.3f}')
axs[0].axline((0,0), slope=1, color='red', lw=1)

axs[1].scatter(tt[:2000], tp[:2000], alpha=0.3, s=2, color='orange')
axs[1].set_xlabel('True Tmax (z-score)'); axs[1].set_title(f'Tmax R²={r2_tmax:.3f}')
axs[1].axline((0,0), slope=1, color='red', lw=1)

epochs_done = TOTAL_EPOCHS - START_EPOCH
axs[2].text(0.5, 0.5, f'Epochs: {START_EPOCH+1}–{TOTAL_EPOCHS}\n'
            f'Best val_loss: {best_val_loss:.4f}\nBest R²_rain: {best_r2_rain:.3f}\nBest R²_tmax: {r2_tmax:.3f}',
            ha='center', va='center', transform=axs[2].transAxes, fontsize=14)
axs[2].axis('off'); axs[2].set_title('Summary')
plt.tight_layout()
plt.savefig('/kaggle/working/eval_plots.png', dpi=150)
plt.show()
print('✓ Plots saved to /kaggle/working/eval_plots.png')


In [ ]:

# ── 10. Package checkpoint for download ───────────────────────────────────────
import os, shutil, time

best_ckpt = '/kaggle/working/vayu_best.pt'
if os.path.exists(best_ckpt):
    info = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    meta = info.get('metadata', {})
    print('=== Best Checkpoint ===')
    print(f'  Epoch     : {meta.get("epoch","?")}')
    print(f'  val_loss  : {meta.get("val_loss","?"):.4f}')
    print(f'  R²_rain   : {meta.get("r2_rain","?"):.4f}')
    print(f'  R²_tmax   : {meta.get("r2_tmax","?"):.4f}')
    print(f'  File size : {os.path.getsize(best_ckpt)/1e6:.1f} MB')
    print(f'\n→ Download vayu_best.pt from Kaggle Output → Data → /kaggle/working/')
    print(f'\nNext step: upload to Kaggle dataset shyam31415/vayu-v2-checkpoint')
    print(f'           then run Stage 2 with TRAINING_MODE = "WARM_V2"')
else:
    print('⚠ No checkpoint found at /kaggle/working/vayu_best.pt')
    print('  Check that training ran successfully and saved at least one checkpoint.')


In [ ]:

# ── 12. Aurora baseline comparison (OPTIONAL — disable during training) ────────
# Microsoft Aurora (1.3B params) is a weather foundation model trained on ERA5.
# We compare its 7-day rainfall/temperature forecasts against VAYU v2 for the
# same Western Ghats region using Open-Meteo as the data bridge.
#
# ⚠ Run this ONLY after training finishes — it needs internet + extra GPU memory.
# ─────────────────────────────────────────────────────────────────────────────

RUN_AURORA = False          # ← set True after training to run comparison

if not RUN_AURORA:
    print('Aurora comparison skipped (RUN_AURORA = False).')
    print('Set RUN_AURORA = True and re-run this cell after training to compare.')
else:
    import urllib.request, json, datetime

    # ── 12a. Install aurora package from Microsoft GitHub ─────────────────────
    !pip install -q git+https://github.com/microsoft/aurora.git
    from aurora import Aurora, Metadata, Batch, AuroraSmall
    import huggingface_hub

    print('Downloading aurora-0.25-small-patch-size weights (~300 MB)...')
    huggingface_hub.hf_hub_download(
        repo_id='microsoft/aurora', filename='aurora-0.25-small-patch-size.ckpt',
        local_dir='/kaggle/working/aurora_ckpt')

    aurora_model = AuroraSmall()
    aurora_model.load_checkpoint(
        '/kaggle/working/aurora_ckpt/aurora-0.25-small-patch-size.ckpt')
    aurora_model = aurora_model.to(DEVICE).eval()
    print(f'✓ Aurora loaded  |  params: {sum(p.numel() for p in aurora_model.parameters())/1e6:.0f}M')

    # ── 12b. Fetch ERA5 surface data via Open-Meteo for Western Ghats ─────────
    # Centroid of our training nodes
    LAT_C, LON_C = 14.5, 75.5          # Approx Western Ghats centre
    TODAY = (datetime.date.today() - datetime.timedelta(days=7)).isoformat()

    url = (f'https://archive-api.open-meteo.com/v1/archive?'
           f'latitude={LAT_C}&longitude={LON_C}'
           f'&start_date={TODAY}&end_date={TODAY}'
           f'&hourly=temperature_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m'
           f'&daily=precipitation_sum,temperature_2m_max,temperature_2m_min'
           f'&timezone=Asia%2FKolkata')
    with urllib.request.urlopen(url) as r:
        era5_data = json.loads(r.read())

    print(f'Open-Meteo ERA5 data fetched for {TODAY}:')
    daily = era5_data.get('daily', {})
    precip = daily.get('precipitation_sum', [None])[0]
    tmax   = daily.get('temperature_2m_max', [None])[0]
    tmin   = daily.get('temperature_2m_min', [None])[0]
    print(f'  Precip: {precip} mm  Tmax: {tmax}°C  Tmin: {tmin}°C')

    # ── 12c. NWP baselines from Open-Meteo (ECMWF IFS + persistence) ──────────
    # We compare: VAYU v2 | ECMWF IFS (Open-Meteo) | persistence
    fc_url = (f'https://api.open-meteo.com/v1/forecast?'
              f'latitude={LAT_C}&longitude={LON_C}'
              f'&daily=precipitation_sum,temperature_2m_max,temperature_2m_min'
              f'&forecast_days=7&timezone=Asia%2FKolkata&models=ecmwf_ifs025')
    with urllib.request.urlopen(fc_url) as r:
        fc_data = json.loads(r.read())

    fc_daily = fc_data.get('daily', {})
    ecmwf_rain = fc_daily.get('precipitation_sum', [])
    ecmwf_tmax = fc_daily.get('temperature_2m_max', [])
    ecmwf_tmin = fc_daily.get('temperature_2m_min', [])

    print('\n=== 7-day forecast comparison (WG centroid) ===')
    print(f'{"Day":<5} {"ECMWF rain":>12} {"ECMWF Tmax":>12} {"ECMWF Tmin":>12}')
    for i, (pr, tx, tn) in enumerate(zip(ecmwf_rain[:7], ecmwf_tmax[:7], ecmwf_tmin[:7]), 1):
        print(f'T+{i:<4} {pr:>12.1f} {tx:>12.1f} {tn:>12.1f}')

    print('\n⚠ Full Aurora inference over Western Ghats grid requires ERA5-format input tensors.')
    print('  See: https://github.com/microsoft/aurora for input format details.')
    print('  VAYU v2 predictions above are from val_loader (IMD-normalised features).')
    print('  Aurora checkpoint downloaded to /kaggle/working/aurora_ckpt/')
